# ST7 Project 2026

## Algorithm
1. Forward solve: m_n → SEM3D → u_sim(x_ric, t) and u(x,t)
2. Misfit: r(t) = u_sim(t) - d_obs(t)
3. Adjoint solve: r(T-t) → SEM3D backward → Λ(x,t)
4. Gradient: g_λ, g_μ from cross-correlation of ε[u] and ε[Λ]
5. CG direction (Fletcher-Reeves): p_n = -g_n + β_n · p_{n-1}
6. Line search (backtracking Armijo): find α_n
7. Update: m_{n+1} = m_n + α_n · p_n

## 0. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import sys
from collections import deque
from mpi4py import MPI
# from pysem import parse_sem3d_traces
from pathlib import Path

# sys.path.append(str(Path("pysem/src").resolve()))

path_to_src = str(Path("pysem/src").resolve())
print(f"Adding {path_to_src} to sys.path")
if path_to_src not in sys.path:
    sys.path.append(path_to_src)

from pysem.parse_sem3d_traces import ParseSEM3DH5Traces
from pysem.generate_h5_materials import write_h5
from pysem.parse_sem3d_snapshots import compute_gradients_main
from util_funct.sbatch_and_wait import sbatch_and_wait
from util_funct.compute_misfit import compute_misfit
from util_funct.read_stations_pos import read_stations_pos
from util_funct.write_backward_spec import write_backward_spec_from_template
from util_funct.write_misfit_files import write_time_reversed_residual_files
from util_funct.compute_dir_parallel import compute_dir_parallel

Adding C:\Users\Nicolò Dal Monte\Desktop\progetto st7\CEISM-Project\pysem\src to sys.path


ModuleNotFoundError: No module named 'matplotlib'

## 1. Paths and parameters

In [ ]:
SEM3D_CONFIG_RES_FOLDER_PATH = "./sem3d_config_files"

FORWARD_PROBLEM_MESHER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "MESHER.sbatch")
FORWARD_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "SOLVER.sbatch")

TRACES_SIMULATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "traces")
TRACES_OSSERVATED_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH, "Uobs")

SEM3D_CONFIG_RES_FOLDER_PATH_ADJ = "./sem3d_config_files_adj"

ADJOINT_SOURCES_FOLDER_NAME = ""    #MUST BE short, otherwise sem3d will complain
ADJOINT_SOURCES_FOLDER_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, ADJOINT_SOURCES_FOLDER_NAME)

STATIONS_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "stations.txt")

BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "template/input_backward_template.spec")
ADJOINT_PROBLEM_SOLVER_SBATCH_PATH = os.path.join(SEM3D_CONFIG_RES_FOLDER_PATH_ADJ, "SOLVER_ADJOINT.sbatch")

In [ ]:
N_ITER = 10 # to be modified
LBFGS_MEM = 5 # Numero di iterazioni da ricordare

# Inizializziamo le code per L-BFGS fuori dal ciclo
y_queue_lam, s_queue_lam = [], []
y_queue_mu,  s_queue_mu  = [], []

# Variabili per memorizzare i chunk precedenti 
m_lam_old_chunk, m_mu_old_chunk = None, None
g_lam_old_chunk, g_mu_old_chunk = None, None

## 2. Load observed data d_obs

In [ ]:
obs_stream = ParseSEM3DH5Traces(
    wkdir=TRACES_OSSERVATED_FOLDER_PATH,
    format='h5',
    names=['Uobs'],
    variables=['Displ'],
    components=['x', 'y', 'z']
)

obs_monitor = obs_stream['Uobs']

## 3. Initial material m_0

In [ ]:
! python3 ./pysem/src/pysem/generate_h5_materials.py @@prop "la" "mu" "ds" @@tag "linear_gradient" @@dir "z" @@xlim -2000 2000 @@ylim -2000 2000 @@zlim -2000 250 @@step 200 200 200 @@pfx 'example'
! mv example* ./sem3d_config_files/forward_problem_step1

## 4. Algorithm

In [ ]:
# sys.argv = ['parse_sem3d_snapshots.py', '@@wkd', './sem3d_config_files/res', '@@begin_time', '0', '@@end_time', '2']
from pysem.parse_sem3d_snapshots import compute_gradients_main

In [ ]:


N_ITER = 10 #to be modified

sbatch_and_wait(FORWARD_PROBLEM_MESHER_SBATCH_PATH)
stations = np.loadtxt(STATIONS_FILE_PATH) #read_stations_pos(STATIONS_FILE_PATH)

for n in range(N_ITER):

    # ── STEP 1 ────────────────────────────────────────
    sbatch_and_wait(FORWARD_PROBLEM_SOLVER_SBATCH_PATH)
    


    # ── STEP 2: MISFIT ────────────────────────────────────────────────────
    J, residual, t_sim, dt_sim = compute_misfit(TRACES_SIMULATED_FOLDER_PATH, obs_monitor)
      
    
    # ── STEP 3 ────────────────────────────────────────

    time_reversed_residual = residual[::-1, :, :]

    file_names = write_time_reversed_residual_files(time_reversed_residual, 
                                                    t_sim, 
                                                    OUTPUT_DIR=ADJOINT_SOURCES_FOLDER_PATH)

    write_backward_spec_from_template(template_backward_spec_path=BACKWARD_INPUT_SPEC_TEMPLATE_FILE_PATH, 
                                      output_backward_spec_path = SEM3D_CONFIG_RES_FOLDER_PATH,
                                      adjoint_sources_folder_path = ADJOINT_SOURCES_FOLDER_PATH, 
                                      stations = stations, 
                                      file_names = file_names)

    sbatch_and_wait(ADJOINT_PROBLEM_SOLVER_SBATCH_PATH)

    # ── STEP 4: GRADIENT ─────────────────────────────────────────────────
    #MPI call --> start parallelization
    
    from mpi4py import MPI
    if not MPI.Is_initialized():
        MPI.Init()
    
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()

    # Carichiamo gli snapshot distribuiti
    snp, snp_adj = GetSnapshots(comm, size, rank)
    
    # Calcolo del gradiente 
    g_lam_chunk, g_mu_chunk, mask = snp.solve_gradients_parallel(snp_adj, dt=dt_sim)


    # ── AGGIORNAMENTO CODE L-BFGS ──────────────────────────────────
    # Estraiamo i chunk del modello corrente per calcolare 's'
    m_lam_chunk = snp.dset['Lamb'][mask] 
    m_mu_chunk  = snp.dset['Mu'][mask]

    if n > 0:
        s_lam = m_lam_chunk - m_lam_old_chunk
        s_mu  = m_mu_chunk  - m_mu_old_chunk
        y_lam = g_lam_chunk - g_lam_old_chunk
        y_mu  = g_mu_chunk  - g_mu_old_chunk
        
        s_queue_lam.insert(0, s_lam)
        y_queue_lam.insert(0, y_lam)
        s_queue_mu.insert(0, s_mu)
        y_queue_mu.insert(0, y_mu)
        
        if len(s_queue_lam) > LBFGS_MEM:
            s_queue_lam.pop()
            y_queue_lam.pop()
            s_queue_mu.pop()
            y_queue_mu.pop()


    # ── STEP 5 (Line 9 screenshot): SEARCH DIRECTIONS ──────────────
    M_len_lam = len(s_queue_lam)
    M_len_mu = len(s_queue_mu)

    dir_lam_chunk = compute_dir_parallel(g_lam_chunk, y_queue_lam, s_queue_lam, M_len_lam, comm)
    dir_mu_chunk  = compute_dir_parallel(g_mu_chunk, y_queue_mu, s_queue_mu, M_len_mu, comm)

    # Salvataggio stato corrente in preparazione al prossimo step e alla prossima iterazione
    m_lam_old_chunk = m_lam_chunk.copy()
    m_mu_old_chunk  = m_mu_chunk.copy()
    g_lam_old_chunk = g_lam_chunk.copy()
    g_mu_old_chunk  = g_mu_chunk.copy()


    # ── STEP 6 (Line 10 screenshot): LINE SEARCH ───────────────────
    
    

    # ── STEP 7 (Line 11 screenshot): UPDATE ────────────────────────
    
    
    
    # --> end parallelization